# University Chatbot - Assistant Académique (RAG Implementation)

Ce notebook contient l'implémentation mise à jour du Chatbot RAG basé sur la **Loi 01.00** et les réglements universitaires. 
Il utilise désormais **Gemini 2.5 Flash** pour la génération et **HuggingFace** pour les embeddings locaux.

In [ ]:
!pip install -q langchain langchain-google-genai faiss-cpu sentence-transformers pypdf python-dotenv

In [ ]:
import os
from pypdf import PdfReader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from google.colab import userdata

# Configuration des API (Secrets Colab ou variables d'environnement)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') if 'GOOGLE_API_KEY' in os.environ else "VOTRE_GOOGLE_API_KEY"

## 1. Prétraitement du Document PDF

In [ ]:
def process_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    raw_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            raw_text += text
    
    # Nettoyage basique
    import re
    raw_text = re.sub(r'\n+', '\n', raw_text)
    raw_text = re.sub(r'\s+', ' ', raw_text)
    
    # Chunking sémantique
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = text_splitter.split_text(raw_text)
    return chunks

# Exemple d'utilisation (adapter le chemin)
# chunks = process_pdf('/content/chatbot_ressources.pdf')

## 2. Création de la Base de Connaissances

In [ ]:
def setup_rag(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_texts(chunks, embeddings)
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2
    )
    
    memory = ConversationBufferWindowMemory(
        memory_key="chat_history",
        k=5,
        return_messages=True
    )
    
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        memory=memory
    )
    return qa_chain

## 3. Conversation avec l'Assistant

In [ ]:
# Initialisation (Simulation)
print("Système prêt pour les tests.")

def ask_question(chain, query):
    result = chain({"question": query})
    return result['answer']